In [2]:
import math
from dataclasses import dataclass
from typing import Literal, Optional, Tuple

import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch import nn

world_size = 1
rank = 0
block_size = 128

@dataclass
class ModelArgs:
    max_batch_size: int = 8
    max_seq_len: int = 4096 * 4
    dtype: Literal["bf16", "fp8"] = "bf16"
    vocab_size: int = 102400
    dim: int = 2048
    inter_dim: int = 10944
    moe_inter_dim: int = 1408
    n_layers: int = 27
    n_dense_layers: int = 1
    n_heads: int = 16
    # mla
    qk_nope_head_dim: int = 128
    qk_rope_head_dim: int = 64
    v_head_dim: int = 128


In [30]:
class RotaryPositionalEncoding(nn.Module):
    def __init__(self, d_head, max_seq_len, base=10000):
        super().__init__()
        theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2).float() / d_head))
        self.register_buffer('theta', theta)
        
        positions = torch.arange(max_seq_len).unsqueeze(1)
        freqs = positions * self.theta.unsqueeze(0)
        
        self.register_buffer('freqs_cis', torch.polar(torch.ones_like(freqs), freqs))
    
    def forward(self, x):
        seq_len = x.shape[2]

        x_complex = x.float().reshape(*x.shape[:-1], -1, 2)
        x_complex = torch.view_as_complex(x_complex)
        
        freqs_cis = self.freqs_cis[:seq_len, :].unsqueeze(0).unsqueeze(0)
        
        x_rotated = x_complex * freqs_cis
        
        x_rotated = torch.view_as_real(x_rotated)
        x_rotated = x_rotated.flatten(3)
        
        return x_rotated.type_as(x)

In [28]:
# 1.0 / (10000 ** (torch.arange(0, 64, 2).float() / 64))

d_head = 8
seq_len = 20

theta = 1.0 / 10000 ** (torch.arange(0, d_head, 2).float() / d_head)
positions = torch.arange(seq_len).unsqueeze(1)


print(theta.shape)
print(positions.shape)

freqs = positions * theta.unsqueeze(0)
print(freqs.shape)
# print(freqs)

freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
print(freqs_cis.shape)

freqs_cis1 = freqs_cis[:6, :].unsqueeze(0).unsqueeze(0)
print(freqs_cis1.shape)
print(freqs_cis1)


torch.Size([4])
torch.Size([20, 1])
torch.Size([20, 4])
torch.Size([20, 4])
torch.Size([1, 1, 6, 4])
tensor([[[[ 1.0000+0.0000j,  1.0000+0.0000j,  1.0000+0.0000j,  1.0000+0.0000j],
          [ 0.5403+0.8415j,  0.9950+0.0998j,  0.9999+0.0100j,  1.0000+0.0010j],
          [-0.4161+0.9093j,  0.9801+0.1987j,  0.9998+0.0200j,  1.0000+0.0020j],
          [-0.9900+0.1411j,  0.9553+0.2955j,  0.9996+0.0300j,  1.0000+0.0030j],
          [-0.6536-0.7568j,  0.9211+0.3894j,  0.9992+0.0400j,  1.0000+0.0040j],
          [ 0.2837-0.9589j,  0.8776+0.4794j,  0.9988+0.0500j,  1.0000+0.0050j]]]])


In [36]:
class DeepSeekAttention(nn.Module):
    def __init__(self, d_model, num_heads, d_latent, d_rope, dropout=0.0, max_seq_len=2048):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_latent = d_latent
        self.d_rope = d_rope

        # content path
        self.w_q = nn.Linear(d_model, d_model)
        self.w_dkv = nn.Linear(d_model, d_latent)
        self.w_uk = nn.Linear(d_latent, d_model)
        self.w_uv = nn.Linear(d_latent, d_model)
        
        # position path
        self.w_k_pos = nn.Linear(d_model, d_rope * num_heads)
        self.w_q_pos = nn.Linear(d_model, d_rope * num_heads)

        self.rope = RotaryPositionalEncoding(d_rope, max_seq_len)        

        # final output projection
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(1, 1, max_seq_len, max_seq_len), diagonal=1).bool())

    def forward(self, x):
        batch_size, seq_len, _ = x.size()

        # content path
        q_c = self.w_q(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        c_kv = self.w_dkv(x)
        k_c = self.w_uk(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        v_c = self.w_uv(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # position path
        q_r_unrotated = self.w_q_pos(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        k_r_unrotated = self.w_k_pos(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        #rotation
        q_r = self.rope(q_r_unrotated)
        k_r = self.rope(k_r_unrotated)

        # combine content and position
        content_score = (q_c @ k_c.transpose(-2, -1)) / (self.d_head ** 0.5)
        position_score = (q_r @ k_r.transpose(-2, -1)) / (self.d_head ** 0.5)
        scores = content_score + position_score

        attn_scores = scores.masked_fill(self.mask[:, :, :seq_len, :seq_len], float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_weights @ v_c).transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.w_o(context_vector)


In [37]:
# --- Usage Example ---
d_model = 512
num_heads = 8
d_latent = 128
d_rope = 64 # Dimension for RoPE, typically d_head or smaller
batch_size = 4
seq_len = 64

# Instantiate the full attention layer
deepseek_attn_layer = DeepSeekAttention(d_model, num_heads, d_latent, d_rope)

# Create a dummy input tensor
dummy_input = torch.randn(batch_size, seq_len, d_model)

# Pass the input through the layer
output = deepseek_attn_layer(dummy_input)

print("DeepSeekAttention Layer successful!")
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

DeepSeekAttention Layer successful!
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])


In [41]:
class Expert(nn.Module):
    def __init__(self, d_model, d_hidden, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class DeepSeekMoE(nn.Module):
    def __init__(self, d_model, n_routed_experts, n_shared_experts, top_k, routed_hidden, shared_hidden, bias_lr, fp16_router=False):
        super().__init__()
        self.top_k = top_k
        self.n_routed_experts = n_routed_experts
        self.n_shared_experts = n_shared_experts

        self.shared_experts = nn.ModuleList([Expert(d_model, shared_hidden) for _ in range(n_shared_experts)])
        self.routed_experts = nn.ModuleList([Expert(d_model, routed_hidden) for _ in range(n_routed_experts)])

        self.register_parameter('centroids', nn.Parameter(torch.empty(n_routed_experts, d_model)))
        nn.init.normal_(self.centroids, std=d_model ** -0.5)

        self.register_buffer('bias', torch.zeros(n_routed_experts))

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        x_flat = x.reshape(-1, d_model)

        shared_out = torch.zeros_like(x)
        for expert in self.shared_experts:
            shared_out += expert(x)

        logits = F.linear(x, self.centroids) + self.bias

        top_logits, topk_index = torch.topk(logits, self.top_k, dim=-1)
        gate = F.softmax(top_logits, dim=-1)

        routed_out = torch.zeros_like(x)
        for i in range(self.n_routed_experts):
            mask = (topk_index == i)

            row_idx, which_idx = mask.nonzero(as_tuple=True)
            if row_idx.numel() == 0:
                continue

            exp_in = x_flat.index_select(0, row_idx)
            out = self.routed_experts[i](exp_in)
            w = gate[row_idx, which_idx].unsqueeze(-1)

            routed_out.index_add_(0, row_idx, out * w)

        routed_out = routed_out.view(batch_size, seq_len, d_model)
        return x + shared_out + routed_out



In [38]:
# =============================================================================
# STAGE 1: SETUP, CONFIGURATION, AND DATA PREPARATION
# =============================================================================

# 1.1. Import Core Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import time
from contextlib import nullcontext
from dataclasses import dataclass

# 1.2. Import Helper Libraries
import numpy as np
from datasets import load_dataset
import tiktoken
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd

print("All dependencies imported successfully.")

All dependencies imported successfully.


In [42]:
import ttnn


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



2026-03-03 14:59:08.907 | DEBUG    | ttnn:<module>:77 - Initial ttnn.CONFIG:
Config{cache_path=/home/ctr-jjovicic/.cache/ttnn,model_cache_path=/home/ctr-jjovicic/.cache/ttnn/models,tmp_dir=/tmp/ttnn,enable_model_cache=false,enable_fast_runtime_mode=true,throw_exception_on_fallback=false,enable_logging=false,enable_graph_report=false,enable_detailed_buffer_report=false,enable_detailed_tensor_report=false,enable_comparison_mode=false,comparison_mode_should_raise_exception=false,comparison_mode_pcc=0.9999,root_report_path=generated/ttnn/reports,report_name=std::nullopt,std::nullopt}


In [43]:
ttnn.cluster.get_cluster_type()

2026-03-03 14:59:19.534 | info     |             UMD | Established firmware bundle version: 18.12.1 (topology_discovery.cpp:418)


ClusterType.N150

2026-03-03 14:59:19.535 | warning  |             UMD | Chip 0 has inconsistent Tensix harvesting information between harvest mask and number of harvested. Board n150 expects 1 units, but harvest mask indicates 2 units. (cluster_descriptor.cpp:1041)
2026-03-03 14:59:19.536 | info     |          Device | Opening user mode device driver (tt_cluster.cpp:223)
2026-03-03 14:59:19.540 | info     |             UMD | Established firmware bundle version: 18.12.1 (topology_discovery.cpp:418)
2026-03-03 14:59:19.541 | warning  |             UMD | Chip 0 has inconsistent Tensix harvesting information between harvest mask and number of harvested. Board n150 expects 1 units, but harvest mask indicates 2 units. (cluster_descriptor.cpp:1041)
2026-03-03 14:59:19.551 | info     |             UMD | Established firmware bundle version: 18.12.1 (topology_discovery.cpp:418)
2026-03-03 14:59:19.552 | warning  |             UMD | Chip 0 has inconsistent Tensix harvesting information between harvest mask and nu